# Challenge 9 — Análisis de Actividad de Producto

Notebook ordenado que cubre: Exploración, Limpieza, Data Quality Report,
Métricas, Visualización, Segmentación y Product Decisions.

#### 0. Configuración Inicial

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Cargar el dataset
df = pd.read_csv('docs/product_activity.csv')

# Guardar el total original antes de cualquier limpieza
ORIGINAL_ROW_COUNT = len(df)
print(f"Dataset cargado: {ORIGINAL_ROW_COUNT} filas, {df.shape[1]} columnas")

---
## 1. Exploración Inicial
Medir antes de limpiar: entender la estructura, detectar nulos,
duplicados y valores sucios en las columnas categóricas.

#### 1.1 — Vista general del dataset
Revisamos las primeras filas, los tipos de dato y la distribución
de las columnas numéricas.

In [ ]:
print("Head")
display(df.head())

print("\nInfo")
df.info()

print("\nDescribe")
display(df.describe())

#### 1.2 — Conteo de nulos y duplicados exactos

In [ ]:
# Nulos por columna
null_counts = df.isnull().sum()
print("Nulos por columna:")
display(null_counts)

# Filas duplicadas exactas
duplicate_count = df.duplicated().sum()
print(f"\nFilas duplicadas exactas: {duplicate_count}")

#### 1.3 — Valores únicos y frecuencias
Revisamos `plan_type`, `post_category` y `device_type` para detectar
variantes sucias (typos, mayúsculas, espacios, valores fuera de diccionario).

In [ ]:
print("plan_type:")
display(df["plan_type"].value_counts())

print("\npost_category:")
display(df["post_category"].value_counts())

print("\ndevice_type:")
display(df["device_type"].value_counts())

#### 1.4 — Chequeos lógicos
Verificamos si hay posts creados antes del signup del usuario y
si la columna `days_since_signup` es consistente con las fechas reales.

In [ ]:
# Convertir fechas temporalmente para el chequeo
signup_dates = pd.to_datetime(df["created_at"], errors="coerce")
post_dates = pd.to_datetime(df["post_created_at"], errors="coerce")

# ¿Cuántos posts ocurren antes del signup?
posts_before_signup_count = (post_dates < signup_dates).sum()
print(f"Posts antes del signup: {posts_before_signup_count}")

# ¿days_since_signup es consistente?
calculated_days = (post_dates - signup_dates).dt.days
original_days = df["days_since_signup"]
date_mismatch_count = (calculated_days != original_days).sum()
print(f"Mismatches en days_since_signup: {date_mismatch_count}")

---
## 2. Limpieza Básica con Criterio
Eliminar duplicados, normalizar columnas categóricas, convertir fechas,
recalcular `days_since_signup` y separar filas con errores duros en cuarentena.

#### 2.1 — Eliminar duplicados exactos

In [ ]:
df = df.drop_duplicates()
print(f"Filas después de eliminar {duplicate_count} duplicados: {len(df)}")

#### 2.2 — Normalización canónica
Mapeamos variantes (typos, mayúsculas, espacios) a los valores canónicos
definidos en el challenge.

In [ ]:
# Paso previo: todo a minúsculas y quitar espacios
df["plan_type"] = df["plan_type"].str.strip().str.lower()
df["device_type"] = df["device_type"].str.strip().str.lower()
df["post_category"] = df["post_category"].str.strip().str.lower()

# Diccionarios de mapeo para variantes conocidas
plan_type_map = {
    "free": "free",
    "pro": "pro",
    "enterprise": "enterprise"
}

device_type_map = {
    "web": "web",
    "mobile": "mobile",
    "desktop": "desktop"
}

category_map = {
    "sport": "sports",
    "sporst": "sports",
    "sp0rts": "sports",
    "tech": "tech",
    "tehc": "tech",
    "life": "life",
    "lfe": "life",
    "gaming": "gaming",
    "gamming": "gaming",
    "music": "music",
    "musc": "music",
    "education": "education",
    "educatoin": "education",
    "health": "health",
    "healt": "health",
    "science": "science",
    "sciense": "science",
    "travel": "travel",
    "trvael": "travel",
    "finance": "finance",
    "finanse": "finance"
}

df["plan_type"] = df["plan_type"].replace(plan_type_map)
df["device_type"] = df["device_type"].replace(device_type_map)
df["post_category"] = df["post_category"].replace(category_map)

# Verificar resultado de la normalización
print("plan_type después de normalizar:")
display(df["plan_type"].value_counts())

print("\ndevice_type después de normalizar:")
display(df["device_type"].value_counts())

print("\npost_category después de normalizar:")
display(df["post_category"].value_counts())

#### 2.3 — Convertir fechas a datetime
Convertimos las columnas de fecha y reportamos las que no se pudieron parsear.

In [ ]:
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df["post_created_at"] = pd.to_datetime(df["post_created_at"], errors="coerce")

# Reportar fechas no parseables
unparseable_signup_count = df["created_at"].isna().sum()
unparseable_post_count = df["post_created_at"].isna().sum()
print(f"Fechas no parseables - signup: {unparseable_signup_count}")
print(f"Fechas no parseables - post: {unparseable_post_count}")

#### 2.4 — Recálculo obligatorio de `days_since_signup`
Creamos `days_since_signup_calc` a partir de las fechas reales y comparamos
con la columna original para detectar inconsistencias.

In [ ]:
# Crear la columna recalculada
df["days_since_signup_calc"] = (df["post_created_at"] - df["created_at"]).dt.days

# Comparar con el original
date_mismatch_count = (df["days_since_signup"] != df["days_since_signup_calc"]).sum()
print(f"Mismatches en days_since_signup: {date_mismatch_count}")

#### 2.5 — Quarantine (Cuarentena)
Separamos las filas con errores duros (post antes de signup, fechas
no parseables, valores fuera de diccionario) en un DataFrame aparte
con su `reason_code`.

In [ ]:
# Condición 1: post creado antes del signup
mask_post_before_signup = df["post_created_at"] < df["created_at"]

# Condición 2: fechas no parseables (NaT)
mask_unparseable_dates = df["created_at"].isna() | df["post_created_at"].isna()

# Condición 3: valores fuera del diccionario fijo
valid_plans = ["free", "pro", "enterprise"]
valid_devices = ["web", "mobile", "desktop"]
valid_categories = ["tech", "life", "sports", "science", "finance",
                    "gaming", "music", "health", "education", "travel"]

mask_invalid_plan = ~df["plan_type"].isin(valid_plans)
mask_invalid_device = ~df["device_type"].isin(valid_devices)
mask_invalid_category = ~df["post_category"].isin(valid_categories)

# Crear columna reason_code concatenando motivos
df["reason_code"] = ""
df.loc[mask_post_before_signup, "reason_code"] += "post_before_signup;"
df.loc[mask_unparseable_dates, "reason_code"] += "unparseable_dates;"
df.loc[mask_invalid_plan, "reason_code"] += "invalid_plan;"
df.loc[mask_invalid_device, "reason_code"] += "invalid_device;"
df.loc[mask_invalid_category, "reason_code"] += "invalid_category;"

# Separar: cuarentena vs core
quarantine_mask = df["reason_code"] != ""
df_quarantine = df[quarantine_mask].copy()
df_core = df[~quarantine_mask].copy()

# Limpiar columna temporal del core
df_core = df_core.drop(columns=["reason_code"])

print(f"Filas en cuarentena: {len(df_quarantine)}")
print(f"Filas en core (limpias): {len(df_core)}")

---
## 3. Data Quality Report
Resumen comparando el dataset RAW original contra el dataset CORE limpio.

In [ ]:
raw_row_count = ORIGINAL_ROW_COUNT
core_row_count = len(df_core)
quarantine_row_count = len(df_quarantine)
quarantine_pct = (quarantine_row_count / raw_row_count) * 100
mismatch_pct = (date_mismatch_count / raw_row_count) * 100

quality_report_df = pd.DataFrame({
    'Métrica': ['Filas RAW (originales)',
                'Duplicados removidos',
                'Filas después de dedup',
                'Filas CORE (limpias)',
                'Filas Quarantine',
                '% Quarantine',
                '% Mismatches en fechas'],
    'Valor': [raw_row_count,
              duplicate_count,
              len(df),
              core_row_count,
              quarantine_row_count,
              f"{quarantine_pct:.2f}%",
              f"{mismatch_pct:.2f}%"]
})

display(quality_report_df)

---
## 4. Métricas y Análisis
A partir del dataset CORE, calculamos distribuciones de volumen,
engagement por segmento, y diferencias entre nivel evento vs usuario.

#### 4.1 — Distribuciones (Volumen)
Usuarios únicos por plan, y actividad (#posts) por país, categoría y dispositivo.

In [ ]:
# Usuarios únicos por plan
print("Usuarios únicos por plan:")
display(df_core.groupby("plan_type")["user_id"].nunique())

# Actividad (#posts) por país
print("\nActividad por país:")
display(df_core["country"].value_counts())

# Actividad (#posts) por categoría
print("\nActividad por categoría:")
display(df_core["post_category"].value_counts())

# Actividad (#posts) por dispositivo
print("\nActividad por dispositivo:")
display(df_core["device_type"].value_counts())

# Gráfico de barras: posts por plan
df_core["plan_type"].value_counts().plot(kind="bar", title="Posts por Plan Type")
plt.ylabel("Cantidad de posts")
plt.show()

#### 4.2 — Engagement (Votos)
Media y mediana de votos recibidos, segmentados por plan, país,
categoría y dispositivo.

In [ ]:
# Votos por plan: media y mediana
print("Votos por plan:")
display(df_core.groupby("plan_type")["votes_received"].agg(["mean", "median"]))

# Votos por país: media y mediana
print("\nVotos por país:")
display(df_core.groupby("country")["votes_received"].agg(["mean", "median"]))

# Votos por categoría: media y mediana
print("\nVotos por categoría:")
display(df_core.groupby("post_category")["votes_received"].agg(["mean", "median"]))

# Votos por dispositivo: media y mediana
print("\nVotos por dispositivo:")
display(df_core.groupby("device_type")["votes_received"].agg(["mean", "median"]))

#### 4.3 — Promedios e Interpretación

**Interpretación:** La unidad de análisis en este dataset es el *evento* (cada fila es un post). Esto significa que un usuario con muchísimos posts impacta fuertemente en el promedio general (sesgo de heavy users o outliers), ya que sus votos se registran repetidamente, distorsionando la realidad del usuario promedio.

In [ ]:
# Promedio de votos por plan
avg_votes_by_plan = df_core.groupby("plan_type")["votes_received"].mean()
print("Promedio de votos por plan:")
display(avg_votes_by_plan)

# Posts promedio por usuario
avg_posts_per_user = df_core.groupby("user_id")["post_id"].count().mean()
print(f"\nPosts promedio por usuario: {avg_posts_per_user:.2f}")

#### 4.4 — Evento vs Usuario

**¿Por qué difieren?** En el nivel *evento*, los usuarios más activos sobrerrepresentan la muestra (si un usuario hace 100 posts, su media pesará 100 veces más). Al calcular por *usuario*, a cada persona se le asigna el mismo peso (1 usuario = 1 promedio), limpiando el efecto que provocan los usuarios extremos.

In [ ]:
# Promedio de votos por FILA (nivel evento)
avg_votes_per_event = df_core["votes_received"].mean()
print(f"Promedio de votos por evento (fila): {avg_votes_per_event:.4f}")

# Promedio de votos agrupado por USUARIO
avg_votes_per_user = df_core.groupby("user_id")["votes_received"].mean().mean()
print(f"Promedio de votos por usuario:       {avg_votes_per_user:.4f}")

---
## 5. Concentración y Temporalidad

#### 5.1 — Concentración (Top 1%)
Evaluamos qué porcentaje de la actividad total (posts y votos) proviene
del 1% de usuarios más activos.

In [ ]:
# Posts y votos por usuario
posts_per_user = df_core.groupby("user_id")["post_id"].count()
votes_per_user = df_core.groupby("user_id")["votes_received"].sum()

# Umbral del top 1%
top1_post_threshold = posts_per_user.quantile(0.99)
top1_vote_threshold = votes_per_user.quantile(0.99)

# Filtrar top 1%
top1_users_by_posts = posts_per_user[posts_per_user >= top1_post_threshold]
top1_users_by_votes = votes_per_user[votes_per_user >= top1_vote_threshold]

# Porcentaje del total
top1_post_pct = (top1_users_by_posts.sum() / posts_per_user.sum()) * 100
top1_vote_pct = (top1_users_by_votes.sum() / votes_per_user.sum()) * 100

print(f"Top 1% usuarios por posts: {len(top1_users_by_posts)} usuarios")
print(f"Top 1% usuarios representan {top1_post_pct:.2f}% de los posts")
print(f"Top 1% usuarios representan {top1_vote_pct:.2f}% de los votos")

#### 5.2 — Tendencia temporal
Actividad (número de posts) y engagement (votos promedio por post)
agrupados por mes.

In [ ]:
# Crear columna de mes
df_core["month"] = df_core["post_created_at"].dt.to_period("M")

# Actividad por mes
monthly_activity = df_core.groupby("month")["post_id"].count()
monthly_activity.plot(kind="line", title="Actividad por Mes")
plt.ylabel("Cantidad de posts")
plt.show()

# Engagement por mes (votos promedio por post)
monthly_engagement = df_core.groupby("month")["votes_received"].mean()
monthly_engagement.plot(kind="line", title="Engagement por Mes")
plt.ylabel("Votos promedio")
plt.show()

---
## 6. Product Decisions (Basadas en evidencia)

#### 6.1 — Preguntas

**¿Qué segmento priorizarías y por qué?**
El segmento Enterprise/Pro, ya que muestran métricas de engagement consistentemente superiores (mayor promedio de votos por post). Además, enfoques de retención hacia usuarios de mobile suelen ser clave para el crecimiento del producto.

**¿Qué parte del tablero "mentía" antes de limpiar?**
La métrica `days_since_signup`. Mostraba un porcentaje importante de mismatches comparado con un recálculo limpio a partir de las fechas reales, generando posibles errores en el cálculo del tiempo de vida (LTV) del usuario.

**¿Qué nuevo dato agregarías al tracking?**
Tiempo promedio leyendo el post o si incluye multimedia, para que el número de votos no sea el único reflejo de valor real (engagement puro).

#### 6.2 — Acciones concretas

**Acción 1 — Campaña de retención:**
El Top 1% de usuarios concentra un porcentaje significativo de posts y votos. Se debe incentivar al resto de la base a publicar, por ejemplo, ofreciendo un "logro/bonus" en el primer mes de uso. *Respaldo: métricas de concentración del Top 1% en sección 5.1.*

**Acción 2 — Mejora de experiencia mobile:**
Si la categoría o dispositivo móvil domina en volumen de actividad, se debe priorizar recursos de ingeniería en optimizar su interfaz en próximas versiones. *Respaldo: distribución de actividad por dispositivo en sección 4.1.*

#### Limitaciones del dataset
- No hay datos de retención ni duración de sesión, limitando la evaluación de engagement real.
- `user_total_posts` es un atributo repetido por fila, lo cual puede sesgar promedios si no se agrupa por usuario.
- No hay información sobre la calidad del contenido del post (longitud, multimedia, etc.).
- El período temporal del dataset puede no ser representativo de tendencias a largo plazo.
- La columna `days_since_signup` original no es confiable; el análisis usa la versión recalculada.

---
## 7. Exportación de Archivos

In [ ]:
# 1. Dataset limpio (core)
df_core.to_csv("clean_product_activity.csv", index=False)
print("Exportado: clean_product_activity.csv")

# 2. Dataset cuarentena (con motivo de exclusión)
df_quarantine.to_csv("quarantine_product_activity.csv", index=False)
print("Exportado: quarantine_product_activity.csv")

# 3. Tabla resumen de métricas clave
metrics_summary_df = pd.DataFrame({
    "Métrica": ["unique_users",
                "total_core_posts",
                "avg_votes_per_event",
                "avg_votes_per_user",
                "top1_post_pct",
                "top1_vote_pct"],
    "Valor": [df_core["user_id"].nunique(),
              len(df_core),
              avg_votes_per_event,
              avg_votes_per_user,
              top1_post_pct,
              top1_vote_pct]
})
metrics_summary_df.to_csv("metrics_summary.csv", index=False)
print("Exportado: metrics_summary.csv")

print("\nExportación completa.")